# Aquatic Center Chatbot — Tester Notebook

This notebook lets you try an experimental chatbot for an aquatic center.

The chatbot can answer questions and manage simple tasks about:

- opening hours;
- prices and subscriptions;
- facility rules;
- course booking, modification, and cancellation;
- spa booking, modification, and cancellation;
- equipment purchases;
- lost item reports.

## Before running

1. Open **Runtime > Change runtime type**.
2. Select **T4 GPU** as hardware accelerator.
3. Click **Save**.
4. Run the cells from top to bottom.

A Hugging Face account is usually **not required** for the default public model. If model loading fails because of access permissions, then a Hugging Face token may be needed by the project owner.

The first startup can take a few minutes because the model may need to be downloaded.


## Suggested things to try

The mock database already contains a few users and bookings. You can use these examples to test realistic scenarios.

| User | Existing data | Example messages to try |
|---|---|---|
| `Mario Rossi` | Course: `swimming_school`, adult, intermediate, Tuesday | `Hi, I am Mario Rossi. I want to cancel my swimming school course.` |
| `Mario Rossi` | Course: `aquagym`, adult, beginner, Wednesday | `I am Mario Rossi. I want to move my aquagym course from Wednesday to Friday.` |
| `Mario Rossi` | Course: `hydrobike`, adult, intermediate, Tuesday | `I'm Mario Rossi. Can I modify my hydrobike course from Tuesday to Thursday?` |
| `Mario Rossi` | Lost item: red goggles, swimming pool, 2026-02-22 | `I am Mario Rossi. I lost red goggles in the swimming pool on 2026-02-22.` |
| `Mario Rossi` | Lost item: blue towel, changing room, 2026-01-15 | `I'm Mario Rossi. I lost a blue towel in the changing room on 2026-01-15.` |
| `Luigi Verdi` | Spa booking: 2026-06-15, 15:30, 2 people | `Hi, I am Luigi Verdi. I want to move my spa booking from 2026-06-15 at 15:30 to 2026-06-16 at 18:00.` |
| `Luigi Verdi` | Spa booking: 2026-06-15, 15:30, 2 people | `I am Luigi Verdi. I want to cancel my spa booking on 2026-06-15 at 15:30.` |

You can also try general questions:

| Area | Example messages |
|---|---|
| Opening hours | `What time does the swimming pool close on Sunday?` |
| Prices | `How much is a monthly gym pass for a student?` |
| Rules | `Are glass bottles allowed at the lido?` |
| Courses | `I want to book a beginner swimming school course for a child.` |
| Spa | `I want to book the spa tomorrow at 18:00 for two people.` |
| Shop | `I want to buy red goggles.` |

Useful chat commands:

| Command | Meaning |
|---|---|
| `reset_state` | Reset only the current conversation state |
| `reset` | Reset both conversation state and mock database |
| `exit` | Stop the chat |


## 1. Mount Google Drive and find the project folder

The notebook is expected to be placed in the root of the project folder.  
However, Colab does not directly expose the notebook path, so this cell searches your Drive for the project folder by looking for the expected project files.

If the folder was shared with you and Colab cannot find it, add a shortcut to your Drive:

**Google Drive > Shared with me > right click the project folder > Add shortcut to Drive**.


In [ ]:
from google.colab import drive
from pathlib import Path
import os
import sys

drive.mount("/content/drive")


def find_project_dir() -> Path:
    marker_files = [
        "colab_utils/requirements_colab.txt",
        "colab_utils/colab_launcher.py",
        "app/chatbot.py",
    ]

    search_roots = [
        Path("/content/drive/MyDrive"),
        Path("/content/drive/Shareddrives"),
    ]

    candidates = []

    for root in search_roots:
        if not root.exists():
            continue

        for requirements_file in root.rglob("requirements_colab.txt"):
            candidate = requirements_file.parents[1]

            if all((candidate / marker).exists() for marker in marker_files):
                candidates.append(candidate)

    candidates = sorted(set(candidates), key=lambda path: str(path).lower())

    if not candidates:
        raise FileNotFoundError(
            "Project folder not found. Make sure the notebook is inside the project folder "
            "or add the shared project folder as a shortcut to your Drive."
        )

    if len(candidates) == 1:
        return candidates[0]

    print("Multiple project folders found:")
    for index, candidate in enumerate(candidates, start=1):
        print(f"{index}. {candidate}")

    choice = input("Choose the project folder number, or press Enter to use the first one: ").strip()
    if not choice:
        return candidates[0]

    return candidates[int(choice) - 1]


PROJECT_DIR = find_project_dir()
COLAB_UTILS_DIR = PROJECT_DIR / "colab_utils"

os.chdir(PROJECT_DIR)

for path in [PROJECT_DIR, COLAB_UTILS_DIR]:
    path_str = str(path)
    if path_str not in sys.path:
        sys.path.insert(0, path_str)

print("Project folder:", PROJECT_DIR)
print("Current directory:", os.getcwd())

## 2. Check GPU

This project is designed to run with a GPU. If this cell fails, change the runtime type to **T4 GPU**, restart the session, and run the notebook again.


In [ ]:
import torch

print("CUDA available:", torch.cuda.is_available())

if not torch.cuda.is_available():
    raise RuntimeError("GPU not detected. Go to Runtime > Change runtime type > T4 GPU, then restart the session.")

print("GPU:", torch.cuda.get_device_name(0))

## 3. Install requirements

This cell installs the Python packages required by the chatbot. It may take a few minutes.


In [ ]:
import os

REQUIREMENTS_PATH = PROJECT_DIR / "colab_utils" / "requirements_colab.txt"
os.environ["REQUIREMENTS_PATH"] = str(REQUIREMENTS_PATH)

print("Installing requirements from:", REQUIREMENTS_PATH)
%pip install -q -r "$REQUIREMENTS_PATH" 

## 4. Load the chatbot

This cell configures the project and loads the default model.

The default model can be changed in `MODEL_NAME`, but testers should usually keep the default value.


In [ ]:
import os
import sys
import warnings
import logging

try:
    from google.colab import userdata
    hf_token = userdata.get("HF_TOKEN")
except Exception:
    hf_token = None

if hf_token:
    os.environ["HF_TOKEN"] = hf_token
    print("HF_TOKEN loaded from Colab Secrets.")
else:
    print("No HF_TOKEN found. Continuing without authentication.")

os.environ["APP_DEBUG"] = "false"
os.environ["PYTHONWARNINGS"] = "ignore::FutureWarning"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=FutureWarning, module=r"bitsandbytes.*")
logging.basicConfig(level=logging.WARNING, format="%(levelname)s:%(message)s", force=True)

os.chdir(PROJECT_DIR)

for path in [PROJECT_DIR, COLAB_UTILS_DIR]:
    path_str = str(path)
    if path_str not in sys.path:
        sys.path.insert(0, path_str)

from colab_launcher import setup_project, load_bot

MODEL_NAME = "qwen3"

setup_project(str(PROJECT_DIR), debug=False)
bot = load_bot(MODEL_NAME)

print("Chatbot loaded successfully.")

## 5. Start the chat

Run the cell below and write your messages in the input box.

When you are done, type:

```text
exit
```


In [ ]:
from colab_launcher import start_chat

start_chat(bot)

## Optional: reset and restart

Run this cell only if you want to clear the conversation and restore the mock database before testing again.


In [ ]:
bot.reset_all()
print("Conversation state and mock database have been reset.")